# Phase 4D — frozen native TRAIN raster audit
Verifies both selected packages, extracts exactly 42 GeoTIFFs for the three Phase 4C-frozen TRAIN pairs, runs the existing native audit, and removes transient rasters. No validation or sealed-test imagery is opened.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_DIR = Path('/kaggle/working/SIH-26167-SATQuery')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', os.environ['SATQUERY_REPO_URL'], str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[multisensor]'], check=True)


In [ ]:
import json
from ml.evaluation.inspect_phase4_native_rasters import inspect_frozen_train_pairs
from ml.evaluation.materialize_phase4_bigearthnet import load_frozen_manifest
from ml.evaluation.phase4_native_audit import FROZEN_MANIFEST_SHA256, FROZEN_SAMPLE_IDS, build_audit_member_allowlists, selective_package_extraction, verify_package_for_audit
EXPERIMENT_DIR = REPO_DIR / 'experiments/phase4_bigearthnet_multisensor'
MANIFEST_PATH = EXPERIMENT_DIR / 'split_manifest.json'
manifest = load_frozen_manifest(MANIFEST_PATH)
allowlists = build_audit_member_allowlists(manifest)
if sum(len(paths) for paths in allowlists.values()) != 42:
    raise RuntimeError('Native audit allowlist must contain exactly 42 rasters')
def find_one(name):
    matches = list(Path('/kaggle/input').rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one attached {name}, found {len(matches)}')
    return matches[0]
packages = {modality: find_one(f'phase4_{modality}_selected.tar.zst') for modality in ('s1', 's2')}
package_manifests = {}
for modality, package in packages.items():
    candidate = package.parent / 'package_manifest.json'
    if not candidate.is_file():
        raise RuntimeError(f'Package manifest is not colocated with {package.name}')
    package_manifests[modality] = candidate
verified_packages = {
    modality: verify_package_for_audit(package, package_manifests[modality], modality=modality, frozen_manifest_sha256=FROZEN_MANIFEST_SHA256)
    for modality, package in packages.items()
}


In [ ]:
OUTPUT_DIR = Path('/kaggle/working/satquery-output') / os.environ['SATQUERY_REMOTE_OUTPUT']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
transient_parent = Path('/kaggle/working/phase4d-native-audit')
with selective_package_extraction(packages, allowlists, transient_parent) as dataset_root:
    extracted_root = dataset_root
    for modality in ('s1', 's2'):
        marker = dataset_root / modality / '_materialization.json'
        marker.write_text(json.dumps({'manifest_sha256': FROZEN_MANIFEST_SHA256, 'source': 'verified_kaggle_package_train_only_audit'}))
    raster_count = sum(1 for path in dataset_root.rglob('*.tif') if path.is_file())
    if raster_count != 42:
        raise RuntimeError(f'Expected exactly 42 extracted rasters, found {raster_count}')
    audit = inspect_frozen_train_pairs(manifest_path=MANIFEST_PATH, dataset_root=dataset_root)
if extracted_root.exists():
    raise RuntimeError('Transient native audit rasters were not cleaned up')
audit_path = OUTPUT_DIR / 'representative_raster_audit.json'
audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + '\n')
native_meta = {
    'schema_version': 1,
    'git_sha': _RUNNER_META['git_sha'],
    'experiment': _RUNNER_META['experiment'],
    'reproducible': _RUNNER_META['reproducible'],
    'dirty_worktree': _RUNNER_META['dirty_worktree'],
    'kernel_sources': ['technobishu/satquery-phase4-materialize-s1', 'technobishu/satquery-phase4-materialize-s2'],
    'frozen_manifest_sha256': FROZEN_MANIFEST_SHA256,
    'frozen_train_sample_ids': list(FROZEN_SAMPLE_IDS),
    'expected_raster_count': 42,
    'opened_splits': ['train'],
    'test_pixels_opened': False,
    'transient_rasters_deleted': True,
    'output_raster_count': 0,
    'verified_packages': {modality: {key: record[key] for key in ('modality', 'package_file', 'package_size_bytes', 'file_count', 'package_sha256', 'manifest_sha256')} for modality, record in verified_packages.items()},
}
(OUTPUT_DIR / 'native_audit_runner_meta.json').write_text(json.dumps(native_meta, indent=2, sort_keys=True) + '\n')
injected_runner_meta = OUTPUT_DIR / 'runner_meta.json'
if injected_runner_meta.exists():
    injected_runner_meta.unlink()
print(json.dumps({'status': audit['status'], 'sample_ids': audit['sample_ids'], 'raster_count': 42}, indent=2))
